# Poisson loss and exposure weights

For counts observed over varying exposure, simulate counts from exposure
times the underlying rate. The response for the model is count divided
by exposure. Pass exposure as a weight to both feature selection and the
downstream Poisson model. Evaluate on held-out rows using the same
exposure-weighted loss.

In [1]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance
from sklearn.model_selection import train_test_split

from arfs.feature_selection import GrootCV

rng = np.random.default_rng(42)
X = pd.DataFrame(rng.normal(size=(800, 3)), columns=["signal", "weak", "noise"])
exposure = rng.uniform(0.2, 1.0, size=len(X))
rate = np.exp(0.6 + 0.9 * X["signal"] + 0.2 * X["weak"])
count = rng.poisson(exposure * rate)
y = count / exposure
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, exposure, random_state=42
)

In [2]:
selector = GrootCV(
    objective="poisson",
    n_folds=3,
    n_iter=1,
    n_jobs=1,
    random_state=42,
    lgbm_params={"num_iterations": 50},
).fit(X_train, y_train, sample_weight=w_train)
model = PoissonRegressor(alpha=0.01).fit(
    selector.transform(X_train), y_train, sample_weight=w_train
)
baseline = DummyRegressor().fit(X_train, y_train, sample_weight=w_train)
predictions = {
    "Training mean rate": baseline.predict(X_test),
    "GrootCV with Poisson regression": model.predict(selector.transform(X_test)),
}
print("Selected columns:", selector.selected_features_.tolist())
pd.Series(
    {
        name: mean_poisson_deviance(y_test, pred, sample_weight=w_test)
        for name, pred in predictions.items()
    },
    name="Exposure-weighted mean Poisson deviance",
).to_frame()

Selected columns: ['signal', 'weak']


,Exposure-weighted mean Poisson deviance
Training mean rate,4.505032
GrootCV with Poisson regression,1.562591


GrootCV's LightGBM fit uses these weights. The SHAP summary still averages
contributions over training rows equally. This example illustrates weight
routing and scoring for one count-generating process.